In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [ ]:
# Données d'exemple : corpus simple
'''
corpus = [
        "Le Maroc est un pays situé en l'afrique de nord. Sa capitale est Rabat. Elle possède une riche histoire.",
        "Rabat est la capitale du Maroc et est connue pour sa culture, son histoire et sa gastronomie.","Marakech est la capitale touristique du Maroc",
        "le Maroc et L'informatique est la science du traitement automatique de l'information par des machines.",
        "L'apprentissage automatique est un sous-domaine de l'intelligence artificielle qui permet aux ordinateurs d'apprendre à partir de données."
    ]
    '''

'\ncorpus = [\n        "Le Maroc est un pays situé en l\'afrique de nord. Sa capitale est Rabat. Elle possède une riche histoire.",\n        "Rabat est la capitale du Maroc et est connue pour sa culture, son histoire et sa gastronomie.","Marakech est la capitale touristique du Maroc",\n        "le Maroc et L\'informatique est la science du traitement automatique de l\'information par des machines.",\n        "L\'apprentissage automatique est un sous-domaine de l\'intelligence artificielle qui permet aux ordinateurs d\'apprendre à partir de données."\n    ]\n    '

In [2]:
corpus = [
    "Ceci est un exemple de phrase.",
    "Un autre exemple de phrase.",
    "Le modèle CBOW est utilisé pour prédire des mots en fonction du contexte.",]

In [3]:
# Prétraitement : tokenisation et création du vocabulaire
def tokenize(corpus):
    tokens = [sentence.lower().split() for sentence in corpus]
    vocab = set([word for sentence in tokens for word in sentence])
    word2idx = {word: idx for idx, word in enumerate(vocab)}
    idx2word = {idx: word for word, idx in word2idx.items()}
    return tokens, word2idx, idx2word


In [4]:
###
tokens, word2idx, idx2word = tokenize(corpus)
print("\nl'espace de Vocabulaire")
print(vocab := list(word2idx.keys()))
print("\nDictionnaire mot to index")
print(word2idx)

print("\nDictionnaire index to mot ")
print(idx2word)


l'espace de Vocabulaire
['utilisé', 'ceci', 'autre', 'le', 'des', 'cbow', 'un', 'de', 'en', 'prédire', 'du', 'phrase.', 'contexte.', 'modèle', 'exemple', 'pour', 'fonction', 'mots', 'est']

Dictionnaire mot to index
{'utilisé': 0, 'ceci': 1, 'autre': 2, 'le': 3, 'des': 4, 'cbow': 5, 'un': 6, 'de': 7, 'en': 8, 'prédire': 9, 'du': 10, 'phrase.': 11, 'contexte.': 12, 'modèle': 13, 'exemple': 14, 'pour': 15, 'fonction': 16, 'mots': 17, 'est': 18}

Dictionnaire index to mot 
{0: 'utilisé', 1: 'ceci', 2: 'autre', 3: 'le', 4: 'des', 5: 'cbow', 6: 'un', 7: 'de', 8: 'en', 9: 'prédire', 10: 'du', 11: 'phrase.', 12: 'contexte.', 13: 'modèle', 14: 'exemple', 15: 'pour', 16: 'fonction', 17: 'mots', 18: 'est'}


In [5]:
### Créer les paires contextes et cible
def create_context_target(tokens, context_size=2):
    context_target_pairs = []
    for sentence in tokens:
        for i in range(context_size,len(sentence)-context_size):
            context=sentence[i-context_size:i] + sentence[i+1:i+1+context_size]
            target=sentence[i]
            context_target_pairs.append((context, target))
    return context_target_pairs

### Encodage des contextes et cibles en indices
def encode_pairs(context_target_pairs, word2idx):
    encoded_pairs=[]
    for context, target in context_target_pairs:
        context_idxs=[word2idx[word] for word in context]
        target_idx=word2idx[target]
        encoded_pairs.append((context_idxs, target_idx))
    return encoded_pairs


In [6]:
context_target_pairs = create_context_target(tokens, context_size=2)
encoded_pairs = encode_pairs(context_target_pairs, word2idx)
print("le contexte associe à la cible")
for i in range(10):
    print(f"Contexte : {context_target_pairs[i][0]}  ;  Cible : {context_target_pairs[i][1]}")

print("\n Le contexte encodé et la cible encodée")
for i in range(10):
    print(f"Contexte idx : {encoded_pairs[i][0]}  ;  Cible idx : {encoded_pairs[i][1]}")

le contexte associe à la cible
Contexte : ['ceci', 'est', 'exemple', 'de']  ;  Cible : un
Contexte : ['est', 'un', 'de', 'phrase.']  ;  Cible : exemple
Contexte : ['un', 'autre', 'de', 'phrase.']  ;  Cible : exemple
Contexte : ['le', 'modèle', 'est', 'utilisé']  ;  Cible : cbow
Contexte : ['modèle', 'cbow', 'utilisé', 'pour']  ;  Cible : est
Contexte : ['cbow', 'est', 'pour', 'prédire']  ;  Cible : utilisé
Contexte : ['est', 'utilisé', 'prédire', 'des']  ;  Cible : pour
Contexte : ['utilisé', 'pour', 'des', 'mots']  ;  Cible : prédire
Contexte : ['pour', 'prédire', 'mots', 'en']  ;  Cible : des
Contexte : ['prédire', 'des', 'en', 'fonction']  ;  Cible : mots

 Le contexte encodé et la cible encodée
Contexte idx : [1, 18, 14, 7]  ;  Cible idx : 6
Contexte idx : [18, 6, 7, 11]  ;  Cible idx : 14
Contexte idx : [6, 2, 7, 11]  ;  Cible idx : 14
Contexte idx : [3, 13, 18, 0]  ;  Cible idx : 5
Contexte idx : [13, 5, 0, 15]  ;  Cible idx : 18
Contexte idx : [5, 18, 15, 9]  ;  Cible idx : 0
Co

In [ ]:
# Tokenisation du corpus et préparation des données
#tokens, word2idx, idx2word = tokenize(corpus)
#context_target_pairs = create_context_target(tokens, context_size=2)
#encoded_pairs = encode_pairs(context_target_pairs, word2idx)


In [8]:
# Définition du modèle CBOW
class CBOWModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, context_size):
        super(CBOWModel, self).__init__()
        self.embeddings=nn.Embedding(vocab_size, embedding_dim)
        self.linear1=nn.Linear(context_size * 2 * embedding_dim, 128)
        self.linear2=nn.Linear(128, vocab_size)

    def forward(self, inputs):
        embeds=self.embeddings(inputs).view((1, -1))
        out=torch.relu(self.linear1(embeds))
        out=self.linear2(out)
        log_probs = torch.log_softmax(out, dim=1)
        return log_probs

# Entraînement du modèle
def train_cbow(model, encoded_pairs, word2idx, idx2word, epochs=100, learning_rate=0.001):
    loss_function=nn.NLLLoss()
    optimizer=optim.SGD(model.parameters(), lr=learning_rate)

    for epoch in range(epochs):
        total_loss = 0
        for context_idxs, target_idx in encoded_pairs:
            context_var=torch.tensor(context_idxs, dtype=torch.long)
            target_var=torch.tensor([target_idx], dtype=torch.long)

            model.zero_grad()
            log_probs=model(context_var)
            loss=loss_function(log_probs, target_var)
            loss.backward()
            optimizer.step()

            total_loss +=loss.item()
        if epoch % 10 == 0:
            print(f'Epoch {epoch}, Loss: {total_loss:.4f}')


In [9]:
# Initialisation et entraînement du modèle
vocab_size=len(word2idx)
embedding_dim=4
context_size=2
model = CBOWModel(vocab_size, embedding_dim, context_size)
train_cbow(model, encoded_pairs, word2idx, idx2word, epochs=100)


Epoch 0, Loss: 34.6279
Epoch 10, Loss: 32.1888
Epoch 20, Loss: 29.8871
Epoch 30, Loss: 27.7077
Epoch 40, Loss: 25.6463
Epoch 50, Loss: 23.6959
Epoch 60, Loss: 21.8513
Epoch 70, Loss: 20.1145
Epoch 80, Loss: 18.4877
Epoch 90, Loss: 16.9701


In [10]:
### la nouvelle representation des mots
embedding_matrix = model.embeddings.weight.detach().numpy()
print(f"\n La nouvelle representation des mots est:\n{embedding_matrix}")
print(f"la taille de vocabulaire est:{len(vocab)}, est la taille de la matrice embedding est :{embedding_matrix.shape}")


 La nouvelle representation des mots est:
[[ 0.69042087  0.86089045 -0.45372987 -1.2370732 ]
 [ 0.2160015  -0.3000538  -1.7465189   0.01978056]
 [-0.2773356  -0.46579394 -0.18851192  0.8327817 ]
 [-0.937779   -1.2027297  -0.6339444   2.2437174 ]
 [ 0.29857817 -0.21193613 -0.35739008  2.004992  ]
 [ 0.9882861  -0.3795885   1.9973139   1.4301952 ]
 [ 0.59246254  2.4849439  -1.0540583  -0.72704923]
 [-0.03950763 -0.19637321 -0.9738786   0.70971054]
 [-0.46924314 -0.929067   -1.6455269  -2.0118303 ]
 [ 1.5672395   1.4914267  -0.7644095   0.61035806]
 [-0.2527917   0.8786643   1.1172777  -1.5499802 ]
 [ 1.4569807   0.33489478  0.4937964   0.9869131 ]
 [ 1.0337185  -0.0561497  -1.1643622  -0.9037515 ]
 [ 0.31359932  1.1569518  -0.8882528  -0.8748663 ]
 [ 0.55678195 -0.19664656 -0.8048229   1.3544989 ]
 [-1.6582757   1.1100012   0.5438848  -0.1137749 ]
 [-0.33455524  0.17742728 -0.395764   -0.13695398]
 [ 0.1145262  -0.19738548  0.31243956  2.1918502 ]
 [ 0.04275895 -0.8763097  -0.08326912  

In [11]:
### la representation de mot ceci
def get_vector(model, word, word2idx):
    idx = word2idx[word]
    return model.embeddings.weight[idx].detach().numpy()

vec_maroc = get_vector(model, "ceci", word2idx)
print(f"la representation du mot ceci est:\n {vec_maroc}")

la representation du mot ceci est:
 [ 0.2160015  -0.3000538  -1.7465189   0.01978056]


In [12]:
### Calculer la similarité des mots avec le mot maroc
def cosine_similarity(vec1, vec2):
    num = np.dot(vec1, vec2)
    den = np.linalg.norm(vec1) * np.linalg.norm(vec2)
    return num / den

v1 = get_vector(model, "ceci", word2idx)
v2 = get_vector(model, "exemple", word2idx)

sim = cosine_similarity(v1, v2)
print("Similarité entre 'ceci' et 'exemple' =", sim)

Similarité entre 'ceci' et 'exemple' = 0.5365244


In [ ]:
# --- Trouver les k mots les plus similaires ---
def top_k_similar(model, word, word2idx, idx2word, k=4):
    target_vec = get_vector(model, word, word2idx)
    similarities = {}

    for other_word in word2idx.keys():
        if other_word == word:
            continue
        vec = get_vector(model, other_word, word2idx)
        sim = cosine_similarity(target_vec, vec)
        similarities[other_word] = sim

    # Trier par similarité décroissante
    sorted_words = sorted(similarities.items(), key=lambda x: x[1], reverse=True)

    return sorted_words[:k]

# --- Exemple : Top 4 mots similaires à "maroc" ---
top4 = top_k_similar(model, "maroc", word2idx, idx2word, k=4)
print("Top 4 mots similaires à 'maroc' :")
for word, sim in top4:
    print(f"{word}  to  similarité = {sim:.4f}")



Top 4 mots similaires à 'maroc' :
aux  to  similarité = 0.6491
un  to  similarité = 0.5202
d'apprendre  to  similarité = 0.5011
marakech  to  similarité = 0.4819


In [ ]:
# Exemple de prédiction
def predict(model, context, word2idx, idx2word):
    context_idxs = torch.tensor([word2idx[word] for word in context], dtype=torch.long)
    with torch.no_grad():
        log_probs = model(context_idxs)
    predicted_idx = torch.argmax(log_probs, dim=1).item()
    return idx2word[predicted_idx]

# Tester le modèle
context = ['le', 'maroc', 'est', 'un']
print(f"Prediction for context {context}: {predict(model, context, word2idx, idx2word)}")


Prediction for context ['le', 'maroc', 'est', 'un']: est
